# Day 2 · 01 · Power BI Semantic Model on Databricks

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

This is the official Power BI semantic-model module. Day 1 built the Gold model; this notebook decides how Power BI should consume it.

## Business Scenario

RetailHub wants to publish an executive Sales dashboard and a drill-through detail page. The Gold model is ready, but the team must still decide source tables, relationships, measures, refresh mode and handoff responsibilities.

## Learning Objectives

By the end of this notebook you can:

- define a Power BI semantic model and its Databricks dependency,
- choose between star schema and wide table consumption,
- validate relationship readiness and freshness,
- decide which logic belongs in Databricks SQL vs DAX or Power Query,
- explain Import, DirectQuery and Composite modes,
- simulate incremental refresh with a half-open interval,
- prepare a practical Power BI handoff packet.

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.

In [ ]:
%run ../../setup/00_setup

### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.

In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")

## 1. Gold Inventory

Definition: a Power BI handoff starts with a concrete object inventory. The BI owner needs object names, row counts and the intended usage of each table.

Expected observation: `fact_sales` and `fact_sales_dashboard` should have the same row count because both are line-grain tables.

In [ ]:
%sql
SELECT 'gold.dim_date' AS object_name, 'date dimension' AS purpose, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'gold.dim_customer', 'customer dimension', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'gold.dim_product', 'product dimension', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'gold.fact_sales', 'star-schema fact at line grain', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'gold.fact_sales_dashboard', 'wide line-grain dashboard table', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name

## 2. Semantic Model Definition

A **Power BI semantic model** is the governed layer that contains tables, relationships, measures, formatting and refresh behavior. Reports query the semantic model; the semantic model queries Databricks.

Databricks Gold should provide trusted rows and reusable fields. Power BI should provide report-specific relationships, measures and user interactions.

Expected observation: semantic modeling is a contract decision, not only a UI step in Power BI Desktop.

## 3. Star Schema vs Wide Table Decision

A **star schema** separates facts and dimensions. It is the governed default for reusable semantic models because dimensions can filter many facts consistently.

A **wide table** repeats dimension attributes on each fact row. It is useful for prototypes or narrow dashboard pages, but it can scan more data in DirectQuery and can duplicate business logic.

Professional rule: use `gold.fact_sales` plus dimensions for governed semantic models; use `gold.fact_sales_dashboard` for quick report prototypes or serving tables.

In [ ]:
%sql
WITH star AS (
  SELECT ROUND(SUM(CASE WHEN f.is_completed THEN f.line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales f
  JOIN gold.dim_customer c ON f.customer_id = c.customer_id
  JOIN gold.dim_product p ON f.product_id = p.product_id
),
wide AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales_dashboard
)
SELECT
  star.revenue AS star_schema_revenue,
  wide.revenue AS wide_table_revenue,
  ROUND(star.revenue - wide.revenue, 2) AS revenue_delta
FROM star CROSS JOIN wide

Expected observation: the revenue delta should be zero. The star schema and wide table are different consumption shapes, not different business definitions.

## 4. Relationship Readiness

**Relationship readiness** means dimension keys are unique and fact rows can match those keys. Power BI many-to-one relationships depend on this condition.

An **orphan check** finds fact rows without matching dimensions. Orphans usually become blanks in slicers and can confuse users.

In [ ]:
%sql
SELECT 'dim_date.date_key uniqueness' AS check_name, COUNT(*) - COUNT(DISTINCT date_key) AS issue_rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer.customer_id uniqueness', COUNT(*) - COUNT(DISTINCT customer_id) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product.product_id uniqueness', COUNT(*) - COUNT(DISTINCT product_id) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales -> dim_date orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales -> dim_customer orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales -> dim_product orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id

Expected observation: every issue count should be zero before building Power BI relationships.

## 5. Databricks SQL vs DAX vs Power Query

| Logic | Best home | Reason |
|---|---|---|
| joins to conformed dimensions | Databricks Gold | shared, expensive and reusable |
| unknown labels and source quality flags | Databricks Gold | consistent across BI tools |
| additive base fields | Databricks Gold fields | easy to validate and reuse |
| report-specific measures | Power BI DAX | interactive and presentation-aware |
| time intelligence formatting | Power BI DAX | depends on report calendar behavior |
| large row-by-row transformations | Databricks SQL | avoid Power Query work over DirectQuery |
| cosmetic report-only renames | Power BI | does not change shared data contract |

Expected observation: the decision is about ownership. Shared logic belongs closer to Gold; visual logic belongs in the semantic model.

## 6. KPI Dictionary and Starter Measures

A **KPI dictionary** defines each measure in business language before report authors build visuals. Ambiguous filters are the fastest way to create inconsistent dashboards.

A **DAX measure** is a reusable calculation evaluated at query time in the current filter context. Measures should express business logic such as revenue, margin rate or return rate.

In [ ]:
%sql
SELECT 'Revenue' AS kpi_name, 'SUM line_revenue where is_completed = true' AS definition, 'Databricks field + DAX measure' AS owner
UNION ALL
SELECT 'Gross Margin', 'SUM line_margin where is_completed = true', 'Databricks field + DAX measure'
UNION ALL
SELECT 'Completed Orders', 'DISTINCTCOUNT order_id where is_completed = true', 'DAX measure'
UNION ALL
SELECT 'Margin Rate %', 'Gross Margin divided by Revenue with divide-by-zero protection', 'DAX measure'
UNION ALL
SELECT 'Return Rate %', 'Returned orders divided by completed plus returned orders', 'DAX measure or certified KPI table' 

Starter DAX measures:

```DAX
Revenue =
CALCULATE (
    SUM ( fact_sales[line_revenue] ),
    fact_sales[is_completed] = TRUE ()
)

Gross Margin =
CALCULATE (
    SUM ( fact_sales[line_margin] ),
    fact_sales[is_completed] = TRUE ()
)

Completed Orders =
CALCULATE (
    DISTINCTCOUNT ( fact_sales[order_id] ),
    fact_sales[is_completed] = TRUE ()
)

Margin Rate % =
DIVIDE ( [Gross Margin], [Revenue] )
```

Expected observation: Databricks supplies governed fields and filters; DAX defines reusable report measures.

## 7. Import, DirectQuery and Composite

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

`Import` copies data into Power BI memory during refresh. It is usually fastest for dashboards and safest for cost.

`DirectQuery` sends each interaction back to the SQL Warehouse. It is useful for live requirements, but it can multiply queries through visuals and slicers.

`Composite` mixes Import and DirectQuery tables. It solves mixed freshness requirements, but it increases model complexity.

In [ ]:
%sql
SELECT 'Executive dashboard' AS report_area, 'Import' AS recommended_mode, 'fast interactions and scheduled refresh' AS reason
UNION ALL
SELECT 'Operational live monitor', 'DirectQuery', 'freshness requirement is stronger than latency risk'
UNION ALL
SELECT 'Hybrid management report', 'Composite', 'import dimensions and aggregates, query live detail only when needed'
UNION ALL
SELECT 'Ad-hoc analyst exploration', 'Import or Databricks SQL', 'avoid uncontrolled DirectQuery fan-out from many visuals' 

## 8. Incremental Refresh Definition

Incremental refresh partitions imported data by a date or timestamp range. Power BI uses `RangeStart` and `RangeEnd` parameters. The safe pattern is a half-open interval: `>= RangeStart` and `< RangeEnd`.

Expected observation: half-open intervals avoid overlapping rows between adjacent refresh windows.

In [ ]:
%sql
WITH refresh_window AS (
  SELECT DATE '2024-01-01' AS range_start, DATE '2024-04-01' AS range_end
)
SELECT
  range_start,
  range_end,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard CROSS JOIN refresh_window
WHERE order_date >= range_start
  AND order_date < range_end
GROUP BY range_start, range_end

## 9. Query Folding Definition

**Query folding** means Power Query can translate transformation steps back to the source query. In Databricks DirectQuery, folding is essential because non-folding transformations can create inefficient or unsupported query plans.

Professional rule: do heavy shaping in Databricks Gold and keep Power Query transformations simple, especially for DirectQuery.

In [ ]:
%sql
EXPLAIN FORMATTED
SELECT
  category,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard
WHERE order_date >= DATE '2024-01-01'
  AND order_date < DATE '2025-01-01'
GROUP BY category
ORDER BY revenue DESC

Expected observation: filters and aggregation should appear in the Databricks SQL plan. If a Power Query step cannot fold, redesign it in Gold or in a serving view.

## 10. Freshness and Row Count Checks

Freshness tells report owners how current the data is. Row-count trend checks catch unexpected drops before a scheduled refresh publishes an empty or partial dataset.

In [ ]:
%sql
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders
FROM gold.fact_sales_dashboard

## 11. Connector Walkthrough

![Power BI connection walkthrough](../../assets/images/powerbi_connection_walkthrough.png)

Power BI needs the Databricks SQL Warehouse hostname and HTTP path. Authentication depends on workspace policy: PAT, OAuth or Microsoft Entra ID.

Trainer note: open SQL Warehouses -> Connection details during delivery and show where those two values come from.

## 12. Power BI Handoff Packet

A professional handoff is a packet of decisions, not just a connection string.

| Handoff item | Required decision |
|---|---|
| Connection details | SQL Warehouse hostname and HTTP path from Databricks UI |
| Authentication | PAT, OAuth or Entra ID pattern chosen by workspace policy |
| Source option | star schema for governed model, dashboard table for quick prototype |
| Relationship map | fact date/customer/product keys to unique dimensions |
| Model mode | Import by default, DirectQuery only with explicit freshness need |
| Refresh expectation | schedule, owner and freshness SLA |
| Cost guardrails | warehouse size, auto-stop, visual count and DirectQuery limits |
| Measure ownership | which measures are DAX-owned vs Gold-owned |

Expected observation: this table is the conversation checklist between Databricks and Power BI owners.

## 13. Performance Analyzer Awareness

Power BI Performance Analyzer shows which visuals trigger slow DAX queries or source queries. For Databricks DirectQuery, pair it with SQL Warehouse Query History.

Expected observation: a slow report page is investigated from both sides: Power BI visual fan-out and Databricks SQL execution profile.

## Summary and Workshop Handoff

You now have the semantic model vocabulary and the contract for Workshop 2: build smaller, refresh-ready, performance-aware objects for Power BI without changing the Day 1 Gold model.